# Práctica 2. Representación algebraica y geométrica de un qubit

**Tema:** Conceptos fundamentales. Superposición cuántica  
**Duración estimada:** 60–75 minutos  
**Modalidad:** Individual  
**Lenguaje:** Python  
**Bibliotecas:** NumPy y Matplotlib  
**Uso de Qiskit:** No se utilizará en esta práctica.

En esta práctica trabajaremos con números complejos, notación de Dirac, productos internos, bases ortonormales, fase global y esfera de Bloch.

## Objetivos

Al finalizar la práctica serás capaz de:

- Calcular conjugados, módulos y fases de números complejos.
- Diferenciar $z^2$ y $|z|^2$.
- Representar *kets* y *bras* con NumPy.
- Calcular productos internos y comprobar la ortogonalidad.
- Validar bases ortonormales.
- Distinguir fase global y fase relativa.

> Completa las celdas marcadas con `TODO` y responde en Markdown a todas las preguntas.

## Entrega

Entrega este notebook con el nombre:

```text
practica_02_representacion_qubit_apellidos_nombre.ipynb
```

Debe contener código ejecutable, resultados, gráficas, respuestas razonadas, pruebas y una conclusión final. No se considerará completa una entrega formada únicamente por código.

## Preparación del entorno

In [1]:
import matplotlib.pyplot as plt
import numpy as np

def applyStyle():
    plt.style.use("default")

    plt.rcParams["font.family"] = "serif"
    plt.rcParams["font.serif"] = ["Times New Roman", "DejaVu Serif", "serif"]
    plt.rcParams["font.size"] = 12
    plt.rcParams["axes.labelsize"] = 12
    plt.rcParams["axes.titlesize"] = 12
    plt.rcParams["xtick.labelsize"] = 10
    plt.rcParams["ytick.labelsize"] = 10
    plt.rcParams["legend.fontsize"] = 10

    plt.rcParams["xtick.direction"] = "in"
    plt.rcParams["ytick.direction"] = "in"
    plt.rcParams["xtick.top"] = True
    plt.rcParams["ytick.right"] = True
    plt.rcParams["xtick.minor.visible"] = True
    plt.rcParams["ytick.minor.visible"] = True

    plt.rcParams["lines.linewidth"] = 1.0
    plt.rcParams["lines.markersize"] = 3

    plt.rcParams["legend.frameon"] = True
    plt.rcParams["legend.edgecolor"] = "black"
    plt.rcParams["legend.fancybox"] = False

COLOR_TARGET = "#5d54a4"  # Blue violet
COLOR_PRED = "#9154a4"    # Purple
COLOR_ERR = "#404040"     # Dark gray

np.set_printoptions(precision=5, suppress=True)

applyStyle()


def isValidState(state, tolerance=1e-10):
    try:
        state = np.asarray(state, dtype=complex)
    except (TypeError, ValueError):  # can it be converted to a complex array?
        return False

    if state.shape != (2,):  # does it have exactly two amplitudes?
        return False

    if not np.all(np.isfinite(state)):  # are all amplitudes finite?
        return False

    norm = np.linalg.norm(state)  # is the norm equal to 1?

    return bool(np.isclose(norm, 1.0, atol=tolerance, rtol=0.0))


def normalizeState(state):
    state = np.asarray(state, dtype=complex)

    if state.shape != (2,):
        raise ValueError("The state must have exactly two amplitudes")

    if not np.all(np.isfinite(state)):
        raise ValueError("The state contains non-finite values")

    norm = np.linalg.norm(state)
    if norm == 0.0:  # only the zero vector has norm exactly 0
        raise ValueError("The zero vector cannot be normalized")

    return state / norm


ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)

# Parte A. Números complejos

## Ejercicio 1. Componentes de un número complejo

Considera:

$$z_1=3+4i,\qquad z_2=1-i,\qquad z_3=-2i,\qquad z_4=-1-\sqrt{3}i.$$

Para cada número muestra su parte real, parte imaginaria, conjugado, módulo, módulo al cuadrado, fase en radianes y fase en grados.

In [2]:
z1 = 3 + 4j
z2 = 1 - 1j
z3 = -2j
z4 = -1 - np.sqrt(3) * 1j

numbers = {"z1": z1, "z2": z2, "z3": z3, "z4": z4}

for name, z in numbers.items():
    phaseRad = np.angle(z)
    print(f"{name} = {z}")
    print(f"  real part  = {z.real:.5f}")
    print(f"  imag part  = {z.imag:.5f}")
    print(f"  conjugate  = {np.conj(z)}")
    print(f"  |z|        = {np.abs(z):.5f}")
    print(f"  |z|^2      = {np.abs(z) ** 2:.5f}")
    print(f"  phase rad  = {phaseRad:.5f}")
    print(f"  phase deg  = {np.degrees(phaseRad):.5f}")
    print()

z1 = (3+4j)
  real part  = 3.00000
  imag part  = 4.00000
  conjugate  = (3-4j)
  |z|        = 5.00000
  |z|^2      = 25.00000
  phase rad  = 0.92730
  phase deg  = 53.13010

z2 = (1-1j)
  real part  = 1.00000
  imag part  = -1.00000
  conjugate  = (1+1j)
  |z|        = 1.41421
  |z|^2      = 2.00000
  phase rad  = -0.78540
  phase deg  = -45.00000

z3 = (-0-2j)
  real part  = -0.00000
  imag part  = -2.00000
  conjugate  = (-0+2j)
  |z|        = 2.00000
  |z|^2      = 4.00000
  phase rad  = -1.57080
  phase deg  = -90.00000

z4 = (-1-1.7320508075688772j)
  real part  = -1.00000
  imag part  = -1.73205
  conjugate  = (-1+1.7320508075688772j)
  |z|        = 2.00000
  |z|^2      = 4.00000
  phase rad  = -2.09440
  phase deg  = -120.00000



**Responde:**

1. ¿Puede el módulo ser negativo?
2. ¿Qué relación existe entre la fase de un número y la de su conjugado?
3. ¿En qué cuadrante se encuentra cada número?
4. ¿Por qué conviene utilizar `np.angle()` en lugar de calcular simplemente $\arctan(b/a)$?

**Respuesta:**

1. No. El módulo es la distancia al origen, |z| = √(a² + b²), y la raíz de una suma de cuadrados nunca sale negativa. Solo vale 0 cuando z = 0.

2. Son opuestas: arg(z*) = −arg(z). Conjugar refleja el número respecto al eje real, así que la fase cambia de signo y el módulo se queda igual. Por ejemplo, `z2` tiene fase −45° y su conjugado +45°.

3. Mirando el signo de la parte real y de la imaginaria:
   - z1 = 3 + 4i está en el cuadrante I (fase 53.13°).
   - z2 = 1 − i está en el cuadrante IV (fase −45°).
   - z3 = −2i no está en ningún cuadrante, cae sobre el eje imaginario negativo (fase −90°).
   - z4 = −1 − √3 i está en el cuadrante III (fase −120°).

4. Porque arctan(b/a) solo devuelve ángulos entre −90° y 90°, y al dividir se pierden los signos: z y −z darían la misma fase. Además falla si a = 0, que es justo el caso de `z3`. `np.angle()` usa arctan2(b, a), que mira las dos partes por separado y devuelve el cuadrante correcto en (−π, π].

## Ejercicio 2. Potencias de $i$

Calcula manualmente $i^{15}$, $i^{26}$, $i^{51}$ e $i^{100}$. Después comprueba los resultados con Python.

In [3]:
powers = [15, 26, 51, 100]
cycle = {0: "1", 1: "i", 2: "-1", 3: "-i"}

for n in powers:
    rest = n % 4
    value = 1j ** n
    print(f"i^{n:<3} -> {n:>3} mod 4 = {rest} | predicted = {cycle[rest]:>2} | 1j**{n} = {value}")

i^15  ->  15 mod 4 = 3 | predicted = -i | 1j**15 = (-0-1j)
i^26  ->  26 mod 4 = 2 | predicted = -1 | 1j**26 = (-1+0j)
i^51  ->  51 mod 4 = 3 | predicted = -i | 1j**51 = (-0-1j)
i^100 -> 100 mod 4 = 0 | predicted =  1 | 1j**100 = (1+0j)


**Responde:** ¿Cada cuántas potencias se repite el patrón? ¿Cómo interviene el resto de dividir el exponente entre 4?

**Respuesta:**

El patrón se repite cada 4 potencias, porque i⁴ = 1. La secuencia es 1, i, −1, −i y vuelve a empezar.

Por eso solo importa el resto de dividir el exponente entre 4, ya que iⁿ = i^(n mod 4):

- i¹⁵: 15 mod 4 = 3, así que i¹⁵ = −i.
- i²⁶: 26 mod 4 = 2, así que i²⁶ = −1.
- i⁵¹: 51 mod 4 = 3, así que i⁵¹ = −i.
- i¹⁰⁰: 100 mod 4 = 0, así que i¹⁰⁰ = 1.

Python da lo mismo. El `-0` que aparece en `(-0-1j)` es cosa de la coma flotante y vale 0.

## Ejercicio 3. Cuadrado y módulo al cuadrado

Para $z=(1+i)/2$, calcula manualmente $z^2$, $z^*$, $z^*z$, $|z|$ y $|z|^2$. Después compruébalo con NumPy.

In [4]:
z = (1 + 1j) / 2

print(f"z      = {z}")
print(f"z^2    = {z ** 2}")
print(f"z*     = {np.conj(z)}")
print(f"z*z    = {np.conj(z) * z}")
print(f"|z|    = {np.abs(z):.5f}")
print(f"|z|^2  = {np.abs(z) ** 2:.5f}")

z      = (0.5+0.5j)
z^2    = 0.5j
z*     = (0.5-0.5j)
z*z    = (0.5+0j)
|z|    = 0.70711
|z|^2  = 0.50000


**Responde:** ¿Por qué $z^2$ no puede utilizarse como probabilidad? ¿Por qué $z^*z$ siempre es real y no negativo? ¿Puede una amplitud tener módulo mayor que 1 en un estado normalizado?

**Respuesta:**

z² no sirve como probabilidad porque en general es un número complejo. Aquí sale z² = 0.5i, un imaginario puro, y una probabilidad tiene que ser un real entre 0 y 1. Además z² cambia con la fase de z.

z*z siempre es real y no negativo porque, si z = a + bi, entonces (a − bi)(a + bi) = a² + b², que es una suma de cuadrados. Es justo |z|², y por eso la regla de Born usa |z|² y no z². Aquí da 0.5.

No, ninguna amplitud puede tener módulo mayor que 1. En un estado normalizado la suma de todos los |zᵢ|² vale 1, y como ningún término es negativo, ninguno puede pasar de 1. Solo llega a 1 cuando el resto de amplitudes son 0.

## Ejercicio 4. Forma cartesiana y polar

Implementa funciones para obtener el módulo y la fase de un número complejo y para reconstruirlo a partir de su forma polar.

In [5]:
def getPolarData(z):
    modulus = np.abs(z)
    phaseRad = np.angle(z)
    phaseDeg = np.degrees(phaseRad)
    return float(modulus), float(phaseRad), float(phaseDeg)


def createComplex(modulus, phase):
    return complex(modulus * np.exp(1j * phase))


polarNumbers = [1 + 1j, -1 + 1j, -1j, 2 * np.exp(1j * np.pi / 3)]

for number in polarNumbers:
    modulus, phaseRad, phaseDeg = getPolarData(number)
    rebuilt = createComplex(modulus, phaseRad)
    print(f"z = {number}")
    print(f"  modulus = {modulus:.5f}")
    print(f"  phase   = {phaseRad:.5f} rad ({phaseDeg:.2f} deg)")
    print(f"  rebuilt = {rebuilt}")
    print(f"  equal   = {np.isclose(number, rebuilt)}")
    print()

z = (1+1j)
  modulus = 1.41421
  phase   = 0.78540 rad (45.00 deg)
  rebuilt = (1.0000000000000002+1j)
  equal   = True

z = (-1+1j)
  modulus = 1.41421
  phase   = 2.35619 rad (135.00 deg)
  rebuilt = (-1+1.0000000000000002j)
  equal   = True

z = (-0-1j)
  modulus = 1.00000
  phase   = -1.57080 rad (-90.00 deg)
  rebuilt = (6.123233995736766e-17-1j)
  equal   = True

z = (1.0000000000000002+1.7320508075688772j)
  modulus = 2.00000
  phase   = 1.04720 rad (60.00 deg)
  rebuilt = (1.0000000000000002+1.7320508075688772j)
  equal   = True



**Responde:** ¿Por qué dos fases que difieren en $2\pi$ representan el mismo número complejo?

**Respuesta:**

Porque la exponencial compleja se repite cada 2π: e^(i(θ+2π)) = e^(iθ) · e^(i2π) = e^(iθ) · 1 = e^(iθ). Sumar 2π a la fase es dar una vuelta completa alrededor del origen, así que el punto acaba donde estaba.

Por eso la fase no es única (vale cualquier θ + 2πk) y `np.angle()` devuelve siempre el valor en (−π, π]. En la salida `createComplex` reconstruye los cuatro números; las diferencias como `1.0000000000000002` o `6e-17` son solo redondeo.

# Parte B. Notación de Dirac y producto interno

## Ejercicio 5. Del *ket* al *bra*

Representa

$$|\psi\rangle=\frac{1}{\sqrt2}\begin{pmatrix}1\\i\end{pmatrix}$$

como array unidimensional y como matriz columna. Obtén el *bra* asociado en ambos casos y muestra sus formas con `shape`.

In [6]:
psiVector = np.array([1, 1j], dtype=complex) / np.sqrt(2)
psiColumn = np.array([[1], [1j]], dtype=complex) / np.sqrt(2)

braVector = np.conj(psiVector)     # only conjugate
braMatrix = psiColumn.conj().T     # conjugate transpose

arrays = {
    "psiVector": psiVector,
    "psiVector.T": psiVector.T,
    "braVector": braVector,
    "psiColumn": psiColumn,
    "braMatrix": braMatrix,
}

for name, array in arrays.items():
    print(name)
    print(f"  shape = {array.shape}")
    print(f"  array =\n{array}")
    print()

psiVector
  shape = (2,)
  array =
[0.70711+0.j      0.     +0.70711j]

psiVector.T
  shape = (2,)
  array =
[0.70711+0.j      0.     +0.70711j]

braVector
  shape = (2,)
  array =
[0.70711-0.j      0.     -0.70711j]

psiColumn
  shape = (2, 1)
  array =
[[0.70711+0.j     ]
 [0.     +0.70711j]]

braMatrix
  shape = (1, 2)
  array =
[[0.70711-0.j      0.     -0.70711j]]



**Responde:**

1. ¿Por qué `psi_vector.T` no cambia la forma del array?
2. ¿Qué operación debe realizarse además de la transposición?
3. ¿Qué formas tienen el *ket* columna y su *bra*?

**Respuesta:**

1. Porque `psiVector` es 1D, con shape `(2,)`. `.T` intercambia ejes y aquí solo hay uno, así que devuelve el mismo array. Para que la traspuesta haga algo el array tiene que ser 2D, como `psiColumn`.

2. Hay que conjugar. El bra es la traspuesta conjugada del ket, ⟨ψ| = |ψ⟩†. Si solo se traspone, la i de la segunda amplitud conserva el signo y los productos internos salen mal. En el caso 1D la traspuesta no hace nada, así que lo que importa de verdad es `np.conj`.

3. El ket columna tiene shape `(2, 1)` (dos filas y una columna) y su bra `(1, 2)` (una fila y dos columnas). Así, fila por columna da una matriz 1 × 1, que es el escalar ⟨ψ|ψ⟩.

## Ejercicio 6. Producto interno

Implementa $\langle a|b\rangle$ sin utilizar inicialmente `np.vdot()`. Recuerda conjugar el primer vector.

In [7]:
def innerProduct(stateA, stateB):
    stateA = np.asarray(stateA, dtype=complex)
    stateB = np.asarray(stateB, dtype=complex)

    return complex(np.sum(np.conj(stateA) * stateB))


cases = [
    ("<0|0>", ket0, ket0),
    ("<1|1>", ket1, ket1),
    ("<0|1>", ket0, ket1),
    ("<1|0>", ket1, ket0),
]

for label, a, b in cases:
    mine = innerProduct(a, b)
    reference = np.vdot(a, b)
    print(f"{label} = {mine}   np.vdot = {reference}   equal: {np.isclose(mine, reference)}")

<0|0> = (1+0j)   np.vdot = (1+0j)   equal: True
<1|1> = (1+0j)   np.vdot = (1+0j)   equal: True
<0|1> = 0j   np.vdot = 0j   equal: True
<1|0> = 0j   np.vdot = 0j   equal: True


## Ejercicio 7. Propiedades del producto interno

Construye

$$|\psi\rangle=\frac{|0\rangle+i|1\rangle}{\sqrt2},\qquad
|\phi\rangle=\frac{|0\rangle-|1\rangle}{\sqrt2}.$$

Comprueba que $\langle\psi|\phi\rangle=\langle\phi|\psi\rangle^*$.

In [8]:
psi = (ket0 + 1j * ket1) / np.sqrt(2)
phi = (ket0 - ket1) / np.sqrt(2)

psiPhi = innerProduct(psi, phi)
phiPsi = innerProduct(phi, psi)

print(f"<psi|phi>  = {psiPhi:.5f}")
print(f"<phi|psi>  = {phiPsi:.5f}")
print(f"<phi|psi>* = {np.conj(phiPsi):.5f}")
print(f"<psi|phi> == <phi|psi>*: {np.isclose(psiPhi, np.conj(phiPsi))}")

<psi|phi>  = 0.50000+0.50000j
<phi|psi>  = 0.50000-0.50000j
<phi|psi>* = 0.50000+0.50000j
<psi|phi> == <phi|psi>*: True


**Responde:** ¿El producto interno debe ser real? ¿Qué propiedad se cumple al intercambiar los estados? ¿Qué error aparece si no se conjuga el primer vector?

**Respuesta:**

No tiene por qué ser real. Aquí ⟨ψ|φ⟩ = 0.5 + 0.5i. Lo que sí es siempre real (y ≥ 0) es el producto de un estado consigo mismo, ⟨ψ|ψ⟩ = ‖ψ‖².

Al intercambiar los estados el resultado se conjuga: ⟨ψ|φ⟩ = ⟨φ|ψ⟩*. En la salida se ve, sale 0.5 + 0.5i en un orden y 0.5 − 0.5i en el otro.

Si no se conjuga el primer vector se calcula un producto normal (como `np.dot`), no el producto interno. Entonces ⟨ψ|ψ⟩ ya no tiene por qué ser real ni positivo. Por ejemplo, para |+i⟩ daría (1 + i²)/2 = 0, como si el estado tuviera norma 0. Además se pierde la simetría conjugada.

# Parte C. Ortogonalidad y bases

## Ejercicio 8. Bases $X$ e $Y$

Construye los estados $|+\rangle$, $|-\rangle$, $|+i\rangle$ y $|-i\rangle$. Para cada uno muestra el vector, la norma, las probabilidades en la base computacional y su fase relativa.

In [9]:
ketPlus = (ket0 + ket1) / np.sqrt(2)
ketMinus = (ket0 - ket1) / np.sqrt(2)
ketPlusI = (ket0 + 1j * ket1) / np.sqrt(2)
ketMinusI = (ket0 - 1j * ket1) / np.sqrt(2)

basisStates = {"|+>": ketPlus, "|->": ketMinus, "|+i>": ketPlusI, "|-i>": ketMinusI}

for name, state in basisStates.items():
    probs = np.abs(state) ** 2
    relativePhase = np.angle(state[1]) - np.angle(state[0])
    print(f"{name}:")
    print(f"  vector         = {state}")
    print(f"  norm           = {np.linalg.norm(state):.5f}")
    print(f"  P(0), P(1)     = {probs[0]:.4f}, {probs[1]:.4f}")
    print(f"  relative phase = {relativePhase:.5f} rad ({np.degrees(relativePhase):.1f} deg)")
    print()

|+>:
  vector         = [0.70711+0.j 0.70711+0.j]
  norm           = 1.00000
  P(0), P(1)     = 0.5000, 0.5000
  relative phase = 0.00000 rad (0.0 deg)

|->:
  vector         = [ 0.70711+0.j -0.70711+0.j]
  norm           = 1.00000
  P(0), P(1)     = 0.5000, 0.5000
  relative phase = 3.14159 rad (180.0 deg)

|+i>:
  vector         = [0.70711+0.j      0.     +0.70711j]
  norm           = 1.00000
  P(0), P(1)     = 0.5000, 0.5000
  relative phase = 1.57080 rad (90.0 deg)

|-i>:
  vector         = [0.70711+0.j      0.     -0.70711j]
  norm           = 1.00000
  P(0), P(1)     = 0.5000, 0.5000
  relative phase = -1.57080 rad (-90.0 deg)



**Responde:** ¿Producen los cuatro estados las mismas probabilidades en la base computacional? ¿Son el mismo estado? ¿Qué información los diferencia? ¿Qué pares deben ser ortogonales?

**Respuesta:**

Sí, los cuatro dan P(0) = P(1) = 0.5, porque sus dos amplitudes tienen módulo 1/√2 y la probabilidad solo depende del módulo.

No son el mismo estado. Coinciden en la base computacional, pero se distinguen al medir en la base X o en la Y.

Los diferencia la fase relativa entre la amplitud de |1⟩ y la de |0⟩: 0° en |+⟩, 180° en |−⟩, 90° en |+i⟩ y −90° en |−i⟩. Esta fase sí es física, a diferencia de la global.

Deben ser ortogonales los pares de cada base: |+⟩ con |−⟩ (base X) y |+i⟩ con |−i⟩ (base Y). En los dos casos la fase relativa difiere en 180°. Un estado de X y uno de Y no son ortogonales, por ejemplo |+⟩ y |+i⟩, como se comprueba en el ejercicio 9.

## Ejercicio 9. Comprobación de ortogonalidad

Implementa una función que valide los estados y determine si su producto interno es aproximadamente cero.

In [10]:
def areOrthogonal(stateA, stateB, tolerance=1e-10):
    if not isValidState(stateA, tolerance) or not isValidState(stateB, tolerance):
        raise ValueError("Both states must be valid normalized states") 

    product = np.vdot(stateA, stateB)
    return bool(np.isclose(np.abs(product), 0.0, atol=tolerance))


pairs = [
    ("|0>, |1>", ket0, ket1),
    ("|+>, |->", ketPlus, ketMinus),
    ("|+i>, |-i>", ketPlusI, ketMinusI),
    ("|0>, |+>", ket0, ketPlus),
    ("|1>, |->", ket1, ketMinus),
    ("|+>, |+i>", ketPlus, ketPlusI),
]

for label, a, b in pairs:
    product = np.vdot(a, b)
    orthogonal = areOrthogonal(a, b)
    print(f"{label:12s} | <a|b> = {product: .5f} | |<a|b>| = {np.abs(product):.5f} | orthogonal = {orthogonal}")

print()

invalidPairs = {
    "notNormalized": ((np.array([2, 0], dtype=complex), ket1), "orthogonal directions, but norm 2 != 1"),
    "wrongShape": ((np.array([1, 0, 0], dtype=complex), ket1), "3 amplitudes"),
}

for name, ((a, b), justification) in invalidPairs.items():
    try:
        areOrthogonal(a, b)
        outcome = "NOT DETECTED"
    except ValueError:
        outcome = "ValueError"
    print(f"{name:15s} -> {outcome:12s} | {justification}")

|0>, |1>     | <a|b> =  0.00000+0.00000j | |<a|b>| = 0.00000 | orthogonal = True
|+>, |->     | <a|b> = -0.00000+0.00000j | |<a|b>| = 0.00000 | orthogonal = True
|+i>, |-i>   | <a|b> =  0.00000+0.00000j | |<a|b>| = 0.00000 | orthogonal = True
|0>, |+>     | <a|b> =  0.70711+0.00000j | |<a|b>| = 0.70711 | orthogonal = False
|1>, |->     | <a|b> = -0.70711+0.00000j | |<a|b>| = 0.70711 | orthogonal = False
|+>, |+i>    | <a|b> =  0.50000+0.50000j | |<a|b>| = 0.70711 | orthogonal = False

notNormalized   -> ValueError   | orthogonal directions, but norm 2 != 1
wrongShape      -> ValueError   | 3 amplitudes


## Ejercicio 10. Base ortonormal

Implementa una función que compruebe que dos estados están normalizados y son ortogonales.

In [11]:
def isOrthonormalBasis(stateA, stateB, tolerance=1e-10):
    if not isValidState(stateA, tolerance) or not isValidState(stateB, tolerance):
        return False

    return areOrthogonal(stateA, stateB, tolerance)


bases = {
    "basisZ": ((ket0, ket1), "normalized and orthogonal"),
    "basisX": ((ketPlus, ketMinus), "normalized and orthogonal"),
    "basisY": ((ketPlusI, ketMinusI), "normalized and orthogonal"),
    "repeatedBasis": ((ket0, ket0), "<0|0> = 1, not orthogonal"),
    "notOrthogonal": ((ket0, ketPlus), "|<0|+>| = 1/sqrt(2) != 0"),
    "notNormalized": ((np.array([1, 1], dtype=complex), ket1), "norm sqrt(2) != 1"),
}

for name, ((a, b), justification) in bases.items():
    valid = isOrthonormalBasis(a, b)
    print(f"{name:15s} -> {str(valid):5s} | {justification}")

basisZ          -> True  | normalized and orthogonal
basisX          -> True  | normalized and orthogonal
basisY          -> True  | normalized and orthogonal
repeatedBasis   -> False | <0|0> = 1, not orthogonal
notOrthogonal   -> False | |<0|+>| = 1/sqrt(2) != 0
notNormalized   -> False | norm sqrt(2) != 1


## Ejercicio 11. Cambio de base

Los coeficientes de $|\psi\rangle$ en una base ortonormal $\{|u\rangle,|v\rangle\}$ son $\langle u|\psi\rangle$ y $\langle v|\psi\rangle$.

In [12]:
def coordinatesInBasis(state, basis):
    u, v = basis

    if not isOrthonormalBasis(u, v):  # <u|psi> and <v|psi> only work for orthonormal bases
        raise ValueError("The basis must be orthonormal")

    return np.array([np.vdot(u, state), np.vdot(v, state)])

basisZ = (ket0, ket1)
basisX = (ketPlus, ketMinus)
basisY = (ketPlusI, ketMinusI)

cases = [
    ("|0> in basis X", ket0, basisX),
    ("|1> in basis X", ket1, basisX),
    ("|+> in basis Z", ketPlus, basisZ),
    ("|-i> in basis Y", ketMinusI, basisY),
]

for label, state, basis in cases:
    coords = coordinatesInBasis(state, basis)
    rebuilt = coords[0] * basis[0] + coords[1] * basis[1]
    print(label)
    print(f"  coordinates = {coords}")
    print(f"  rebuilt     = {rebuilt}")
    print(f"  equal       = {np.allclose(state, rebuilt)}")
    print()

try:
    coordinatesInBasis(ket0, (ket0, ketPlus))
    outcome = "NOT DETECTED"
except ValueError:
    outcome = "ValueError"
print(f"basis (|0>, |+>) -> {outcome} | not orthogonal, it is not a basis for this formula")

|0> in basis X
  coordinates = [0.70711+0.j 0.70711+0.j]
  rebuilt     = [1.+0.j 0.+0.j]
  equal       = True

|1> in basis X
  coordinates = [ 0.70711+0.j -0.70711+0.j]
  rebuilt     = [0.+0.j 1.+0.j]
  equal       = True

|+> in basis Z
  coordinates = [0.70711+0.j 0.70711+0.j]
  rebuilt     = [0.70711+0.j 0.70711+0.j]
  equal       = True

|-i> in basis Y
  coordinates = [0.+0.j 1.+0.j]
  rebuilt     = [0.70711+0.j      0.     -0.70711j]
  equal       = True

basis (|0>, |+>) -> ValueError | not orthogonal, it is not a basis for this formula


**Responde:** ¿Cambiar de base modifica el estado físico? ¿Qué cambia? ¿Qué coordenadas debe tener $|-i\rangle$ en la base $Y$?

**Respuesta:**

No. El estado físico es el mismo, solo cambia cómo lo escribimos. Es como describir un vector con otros ejes (el vector no se mueve). Se comprueba al reconstruirlo, `np.allclose` da `True` en todos los casos.

Lo que cambia son las coordenadas, los coeficientes ⟨u|ψ⟩ y ⟨v|ψ⟩. Por ejemplo, |0⟩ es (1, 0) en la base Z pero (1/√2, 1/√2) en la base X.

|−i⟩ debe tener coordenadas (0, 1) en la base Y, porque es justo el segundo vector de la base: ⟨+i|−i⟩ = 0 porque son ortogonales y ⟨−i|−i⟩ = 1 porque está normalizado. Es lo que sale en la salida.

# Parte D. Fase global y fase relativa

## Ejercicio 12. Comparación de estados

Considera:

$$|\psi_A\rangle=\frac{1}{\sqrt2}(1,i)^T,\quad
|\psi_B\rangle=\frac{1}{\sqrt2}(i,-1)^T,$$

$$|\psi_C\rangle=\frac{1}{\sqrt2}(1,-i)^T,\quad
|\psi_D\rangle=-\frac{1}{\sqrt2}(1,i)^T.$$

Comprueba sus normas y probabilidades. Calcula el producto interno de $A$ con cada estado y determina cuáles son físicamente equivalentes a $A$.

In [13]:
psiA = np.array([1, 1j], dtype=complex) / np.sqrt(2)
psiB = np.array([1j, -1], dtype=complex) / np.sqrt(2)
psiC = np.array([1, -1j], dtype=complex) / np.sqrt(2)
psiD = -np.array([1, 1j], dtype=complex) / np.sqrt(2)

phaseStates = {"psiA": psiA, "psiB": psiB, "psiC": psiC, "psiD": psiD}

for name, state in phaseStates.items():
    probs = np.abs(state) ** 2
    product = np.vdot(psiA, state)
    print(f"{name}: {state}")
    print(f"  norm = {np.linalg.norm(state):.5f} | P(0) = {probs[0]:.4f}, P(1) = {probs[1]:.4f}")
    print(f"  <A|{name[-1]}> = {product: .5f} | |<A|{name[-1]}>| = {np.abs(product):.5f}")
    print(f"  equivalent to A = {np.isclose(np.abs(product), 1.0)}")
    print()

psiA: [0.70711+0.j      0.     +0.70711j]
  norm = 1.00000 | P(0) = 0.5000, P(1) = 0.5000
  <A|A> =  1.00000+0.00000j | |<A|A>| = 1.00000
  equivalent to A = True

psiB: [ 0.     +0.70711j -0.70711+0.j     ]
  norm = 1.00000 | P(0) = 0.5000, P(1) = 0.5000
  <A|B> =  0.00000+1.00000j | |<A|B>| = 1.00000
  equivalent to A = True

psiC: [ 0.70711+0.j      -0.     -0.70711j]
  norm = 1.00000 | P(0) = 0.5000, P(1) = 0.5000
  <A|C> =  0.00000+0.00000j | |<A|C>| = 0.00000
  equivalent to A = False

psiD: [-0.70711+0.j      -0.     -0.70711j]
  norm = 1.00000 | P(0) = 0.5000, P(1) = 0.5000
  <A|D> = -1.00000+0.00000j | |<A|D>| = 1.00000
  equivalent to A = True



**Responde:** ¿Qué factor relaciona $A$ con $B$? ¿Y $A$ con $D$? ¿Por qué $A$ y $C$ no son equivalentes aunque produzcan las mismas probabilidades?

**Respuesta:**

A y B se relacionan por un factor i. Si multiplicamos A = (1, i)/√2 por i sale (i, i²)/√2 = (i, −1)/√2, que es B. Es una fase global de 90°, y por eso |⟨A|B⟩| = 1.

A y D se relacionan por un factor −1 = e^(iπ), es decir, D = −A. También es una fase global y también da |⟨A|D⟩| = 1.

A y C no son equivalentes porque en C el cambio no afecta a las dos componentes por igual, solo cambia el signo de la segunda (i pasa a −i). Eso es una fase relativa, no global. Las probabilidades en la base computacional coinciden (0.5 y 0.5), pero |⟨A|C⟩| = 0, así que no solo son distintos, sino ortogonales. De hecho A es |+i⟩ y C es |−i⟩.

## Ejercicio 13. Equivalencia salvo fase global

Para estados normalizados, utiliza el criterio $|\langle\psi|\phi\rangle|=1$. No dividas componentes, porque alguna podría ser cero.

In [14]:
def areGloballyEquivalent(stateA, stateB, tolerance=1e-10):
    if not isValidState(stateA, tolerance) or not isValidState(stateB, tolerance):
        raise ValueError("Both states must be valid normalized states")  # |<a|b>| = 1 only works with norm 1

    product = np.vdot(stateA, stateB)
    return bool(np.isclose(np.abs(product), 1.0, atol=tolerance))


cases = {
    "A vs B": ((psiA, psiB), "B = i*A (global phase)"),
    "A vs D": ((psiA, psiD), "D = -A (global phase)"),
    "A vs C": ((psiA, psiC), "relative phase"),
    "|+> vs -|+>": ((ketPlus, -ketPlus), "global phase -1"),
    "|0> vs |1>": ((ket0, ket1), "orthogonal"),
    "|+> vs |->": ((ketPlus, ketMinus), "relative phase"),
}

for name, ((a, b), justification) in cases.items():
    equivalent = areGloballyEquivalent(a, b)
    print(f"{name:12s} -> {str(equivalent):5s} | {justification}")

print()

try:
    areGloballyEquivalent(np.array([2, 0], dtype=complex), ket0)
    outcome = "NOT DETECTED"
except ValueError:
    outcome = "ValueError"
print(f"{'[2, 0] vs |0>':12s} -> {outcome:12s} | norm 2, the criterion needs normalized states")

A vs B       -> True  | B = i*A (global phase)
A vs D       -> True  | D = -A (global phase)
A vs C       -> False | relative phase
|+> vs -|+>  -> True  | global phase -1
|0> vs |1>   -> False | orthogonal
|+> vs |->   -> False | relative phase

[2, 0] vs |0> -> ValueError   | norm 2, the criterion needs normalized states


**Responde:** ¿Por qué es problemático dividir una componente entre otra? ¿Por qué el módulo del producto interno elimina la fase global? ¿Puede aplicarse directamente el criterio a vectores no normalizados?

**Respuesta:**

Dividir componentes es problemático porque alguna puede ser 0, como la segunda de |0⟩ = (1, 0), y la división falla. Si es muy pequeña, el cociente amplifica los errores de redondeo. Con el módulo del producto interno no hace falta dividir nada.

El módulo elimina la fase global porque si |φ⟩ = e^(iα)|ψ⟩, entonces ⟨ψ|φ⟩ = e^(iα)⟨ψ|ψ⟩ = e^(iα), y |e^(iα)| = 1 para cualquier α. Si la diferencia es de fase relativa, el módulo sale menor que 1 (por Cauchy–Schwarz), como pasa con A y C.

No se puede aplicar tal cual a vectores sin normalizar. En ese caso el máximo de |⟨ψ|φ⟩| es ‖ψ‖·‖φ‖, no 1. Habría que normalizar antes o comparar con ‖ψ‖·‖φ‖. Por eso `areGloballyEquivalent` comprueba primero con `isValidState` que los dos tengan norma 1 y, si no, lanza `ValueError`. Con [2, 0] y |0⟩ sale `ValueError`, aunque son el mismo estado salvo normalización, en vez de un `False` que sería engañoso.

## Pruebas mínimas

In [15]:
def raises(errorType, func, *args, **kwargs):
    try:
        func(*args, **kwargs)
        return False

    except errorType:
        return True

assert np.isclose(np.abs(z1), 5.0)
assert np.isclose(np.degrees(np.angle(z4)), -120.0)
assert np.isclose(1j ** 15, -1j)
assert np.isclose(1j ** 100, 1)
assert np.isclose(z ** 2, 0.5j)                                        # z² is complex
assert np.isclose(np.conj(z) * z, np.abs(z) ** 2)                      # z*z = |z|²
assert np.isclose(createComplex(*getPolarData(-1 + 1j)[:2]), -1 + 1j)
assert psiColumn.shape == (2, 1)
assert braMatrix.shape == (1, 2)
assert np.isclose(innerProduct(psi, phi), np.conj(innerProduct(phi, psi)))
assert np.isclose(innerProduct(psi, phi), np.vdot(psi, phi))
assert np.isclose(innerProduct(ketPlusI, ketPlusI), 1.0)
assert areOrthogonal(ketPlus, ketMinus)
assert areOrthogonal(ketPlusI, ketMinusI)
assert not areOrthogonal(ket0, ketPlus)
assert isOrthonormalBasis(ket0, ket1)
assert not isOrthonormalBasis(ket0, ket0)
assert not isOrthonormalBasis(np.array([1, 1]), ket1)
assert np.allclose(coordinatesInBasis(ketMinusI, basisY), [0, 1])
assert raises(ValueError, areOrthogonal, np.array([2, 0]), ket1)       # not normalized
assert raises(ValueError, coordinatesInBasis, ket0, (ket0, ketPlus))   # not an orthonormal basis
assert areGloballyEquivalent(psiA, psiB)
assert areGloballyEquivalent(psiA, psiD)
assert not areGloballyEquivalent(psiA, psiC)
assert raises(ValueError, areGloballyEquivalent, np.array([2, 0]), ket0)
assert raises(ValueError, normalizeState, [0, 0])